# Query 7: Join Trip Data with Borough Lookup Table
**Type:** Broadcast Join vs Sort-Merge Join
**Problem:** Enrich trips with borough names by joining with a small lookup table built from coordinate ranges. Compares Broadcast vs Sort-Merge join strategies.

In [1]:
import time
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import broadcast

spark = SparkSession.builder \
    .appName('Q7_BroadcastJoin') \
    .master('local[*]') \
    .config('spark.sql.shuffle.partitions', '8') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

26/04/25 18:12:53 WARN Utils: Your hostname, mariam-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/04/25 18:12:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 18:12:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/25 18:12:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 3.5.1


In [2]:
DATA_PATH = '../data/yellow_tripdata_2015-01.csv'

df = spark.read.option('header','true').option('inferSchema','true').csv(DATA_PATH)
df = df.withColumnRenamed('fare_amount','fare') \
       .withColumn('lat_grid', F.round('pickup_latitude', 1)) \
       .withColumn('lon_grid', F.round('pickup_longitude', 1))

df.createOrReplaceTempView('trips')

# Build a small borough lookup table using NYC coordinate ranges
borough_data = [
    (40.7, -74.0, 'Manhattan'),
    (40.6, -73.9, 'Brooklyn'),
    (40.7, -73.8, 'Queens'),
    (40.8, -73.9, 'Bronx'),
    (40.6, -74.1, 'Staten Island'),
    (40.7, -73.9, 'Manhattan'),
    (40.8, -74.0, 'Manhattan'),
    (40.6, -74.0, 'Brooklyn'),
]
borough_schema = ['lat_grid','lon_grid','borough']
borough_df = spark.createDataFrame(borough_data, borough_schema)
borough_df.createOrReplaceTempView('boroughs')

print(f'Trips: {df.count():,} | Borough lookup: {borough_df.count()} rows')
rdd = df.rdd

Trips: 11,157,879 | Borough lookup: 8 rows


## RDD Implementation (manual broadcast)

In [8]:
start = time.time()

borough_map = {(row['lat_grid'], row['lon_grid']): row['borough']
               for row in borough_df.collect()}
bc = spark.sparkContext.broadcast(borough_map)

def safe_float(val):
    try:
        return round(float(val), 1)
    except (TypeError, ValueError):
        return None

result_rdd = (
    rdd
    .filter(lambda r: r['pickup_latitude']  is not None
                  and r['pickup_longitude'] is not None
                  and r['fare']             is not None)
    .map(lambda r: {
        'fare': r['fare'],
        'pickup_datetime': r['tpep_pickup_datetime'],
        'lat': safe_float(r['pickup_latitude']),
        'lon': safe_float(r['pickup_longitude'])
    })
    .filter(lambda r: r['lat'] is not None and r['lon'] is not None)
    .map(lambda r: {
        'fare': r['fare'],
        'pickup_datetime': r['pickup_datetime'],
        'borough': bc.value.get((r['lat'], r['lon']), 'Unknown')
    })
)

rdd_count = result_rdd.count()
rdd_time  = time.time() - start
print(f'RDD (manual broadcast) | Enriched: {rdd_count:,} | Time: {rdd_time:.2f}s')

RDD (manual broadcast) | Enriched: 11,157,877 | Time: 88.03s


## DataFrame – Broadcast Join

In [4]:
start = time.time()

result_bj = (
    df.join(broadcast(borough_df),
            on=['lat_grid','lon_grid'], how='left')
      .select('fare','tpep_pickup_datetime','borough',
              'pickup_latitude','pickup_longitude')
)
print('--- Broadcast Join Plan ---')
result_bj.explain(True)
bj_count = result_bj.count()
bj_time  = time.time() - start
print(f'Broadcast Join | Count: {bj_count:,} | Time: {bj_time:.2f}s')
result_bj.show(10)

--- Broadcast Join Plan ---
== Parsed Logical Plan ==
'Project ['fare, 'tpep_pickup_datetime, 'borough, 'pickup_latitude, 'pickup_longitude]
+- Project [lat_grid#76, lon_grid#97, VendorID#17, tpep_pickup_datetime#18, tpep_dropoff_datetime#19, passenger_count#20, trip_distance#21, pickup_longitude#22, pickup_latitude#23, RateCodeID#24, store_and_fwd_flag#25, dropoff_longitude#26, dropoff_latitude#27, payment_type#28, fare#55, extra#30, mta_tax#31, tip_amount#32, tolls_amount#33, improvement_surcharge#34, total_amount#35, borough#121]
   +- Join LeftOuter, ((lat_grid#76 = lat_grid#119) AND (lon_grid#97 = lon_grid#120))
      :- Project [VendorID#17, tpep_pickup_datetime#18, tpep_dropoff_datetime#19, passenger_count#20, trip_distance#21, pickup_longitude#22, pickup_latitude#23, RateCodeID#24, store_and_fwd_flag#25, dropoff_longitude#26, dropoff_latitude#27, payment_type#28, fare#55, extra#30, mta_tax#31, tip_amount#32, tolls_amount#33, improvement_surcharge#34, total_amount#35, lat_grid#7

Broadcast Join | Count: 11,157,879 | Time: 11.18s
+----+--------------------+---------+------------------+-------------------+
|fare|tpep_pickup_datetime|  borough|   pickup_latitude|   pickup_longitude|
+----+--------------------+---------+------------------+-------------------+
|12.0| 2015-01-15 19:05:39|Manhattan|  40.7501106262207|   -73.993896484375|
|14.5| 2015-01-10 20:33:38|Manhattan|  40.7242431640625| -74.00164794921875|
| 9.5| 2015-01-10 20:33:38|Manhattan| 40.80278778076172|-73.963340759277344|
| 3.5| 2015-01-10 20:33:39|Manhattan| 40.71381759643555|-74.009086608886719|
|15.0| 2015-01-10 20:33:39|Manhattan|40.762428283691406|-73.971176147460938|
|27.0| 2015-01-10 20:33:39|    Bronx|  40.7740478515625|-73.874374389648438|
|14.0| 2015-01-10 20:33:39|Manhattan|40.726009368896484|  -73.9832763671875|
| 7.0| 2015-01-10 20:33:39|Manhattan|  40.7341423034668|-74.002662658691406|
|52.0| 2015-01-10 20:33:39|     NULL| 40.64435577392578|-73.783042907714844|
| 6.5| 2015-01-10 20:33:40

## DataFrame – Sort-Merge Join (no broadcast hint)

In [5]:
start = time.time()
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')

result_smj = (
    df.join(borough_df, on=['lat_grid','lon_grid'], how='left')
      .select('fare','tpep_pickup_datetime','borough',
              'pickup_latitude','pickup_longitude')
)
print('--- Sort-Merge Join Plan ---')
result_smj.explain(True)
smj_count = result_smj.count()
smj_time  = time.time() - start
print(f'Sort-Merge Join | Count: {smj_count:,} | Time: {smj_time:.2f}s')

spark.conf.set('spark.sql.autoBroadcastJoinThreshold','10485760')

--- Sort-Merge Join Plan ---
== Parsed Logical Plan ==
'Project ['fare, 'tpep_pickup_datetime, 'borough, 'pickup_latitude, 'pickup_longitude]
+- Project [lat_grid#76, lon_grid#97, VendorID#17, tpep_pickup_datetime#18, tpep_dropoff_datetime#19, passenger_count#20, trip_distance#21, pickup_longitude#22, pickup_latitude#23, RateCodeID#24, store_and_fwd_flag#25, dropoff_longitude#26, dropoff_latitude#27, payment_type#28, fare#55, extra#30, mta_tax#31, tip_amount#32, tolls_amount#33, improvement_surcharge#34, total_amount#35, borough#121]
   +- Join LeftOuter, ((lat_grid#76 = lat_grid#119) AND (lon_grid#97 = lon_grid#120))
      :- Project [VendorID#17, tpep_pickup_datetime#18, tpep_dropoff_datetime#19, passenger_count#20, trip_distance#21, pickup_longitude#22, pickup_latitude#23, RateCodeID#24, store_and_fwd_flag#25, dropoff_longitude#26, dropoff_latitude#27, payment_type#28, fare#55, extra#30, mta_tax#31, tip_amount#32, tolls_amount#33, improvement_surcharge#34, total_amount#35, lat_grid#

Sort-Merge Join | Count: 11,157,879 | Time: 15.55s


## Spark SQL with BROADCAST hint

In [6]:
start = time.time()
result_sql = spark.sql("""
    SELECT /*+ BROADCAST(b) */
           t.fare, t.tpep_pickup_datetime,
           b.borough,
           t.pickup_latitude, t.pickup_longitude
    FROM   trips   t
    LEFT JOIN boroughs b
           ON  t.lat_grid = b.lat_grid
           AND t.lon_grid = b.lon_grid
    ORDER BY t.fare DESC
    LIMIT 100000
""")
result_sql.explain(True)
sql_count = result_sql.count()
sql_time  = time.time() - start
print(f'SQL (BROADCAST hint) | Count: {sql_count:,} | Time: {sql_time:.2f}s')
result_sql.show(10)

== Parsed Logical Plan ==
'GlobalLimit 100000
+- 'LocalLimit 100000
   +- 'Sort ['t.fare DESC NULLS LAST], true
      +- 'UnresolvedHint BROADCAST, ['b]
         +- 'Project ['t.fare, 't.tpep_pickup_datetime, 'b.borough, 't.pickup_latitude, 't.pickup_longitude]
            +- 'Join LeftOuter, (('t.lat_grid = 'b.lat_grid) AND ('t.lon_grid = 'b.lon_grid))
               :- 'SubqueryAlias t
               :  +- 'UnresolvedRelation [trips], [], false
               +- 'SubqueryAlias b
                  +- 'UnresolvedRelation [boroughs], [], false

== Analyzed Logical Plan ==
fare: double, tpep_pickup_datetime: timestamp, borough: string, pickup_latitude: double, pickup_longitude: string
GlobalLimit 100000
+- LocalLimit 100000
   +- Sort [fare#55 DESC NULLS LAST], true
      +- Project [fare#55, tpep_pickup_datetime#18, borough#121, pickup_latitude#23, pickup_longitude#22]
         +- Join LeftOuter, ((lat_grid#76 = lat_grid#119) AND (lon_grid#97 = lon_grid#120))
            :- SubqueryAlia

SQL (BROADCAST hint) | Count: 100,000 | Time: 88.50s


+------+--------------------+---------+------------------+-------------------+
|  fare|tpep_pickup_datetime|  borough|   pickup_latitude|   pickup_longitude|
+------+--------------------+---------+------------------+-------------------+
|3005.5| 2015-01-02 20:06:34|Manhattan|40.711856842041016|-74.014335632324219|
|999.99| 2015-01-23 11:15:00|     NULL|               0.0|                  0|
|999.99| 2015-01-16 14:48:00|     NULL|               0.0|                  0|
| 980.0| 2015-01-28 08:54:07|    Bronx|40.775882720947266|-73.921890258789063|
| 965.0| 2015-01-17 05:54:37|   Queens| 40.69195556640625|-73.811416625976563|
| 965.0| 2015-01-09 17:42:08|     NULL| 40.68599319458008|-73.591827392578125|
|949.99| 2015-01-07 13:00:53|     NULL| 40.68608093261719|-73.591835021972656|
| 900.0| 2015-01-20 07:07:40|    Bronx|40.778804779052734| -73.91448974609375|
| 900.0| 2015-01-08 11:47:20|Manhattan| 40.71745681762695|-73.993446350097656|
| 900.0| 2015-01-07 16:34:56|Manhattan|40.7181434631

## Performance Comparison

In [9]:
print('='*65)
print(f'{"Metric":<25} {"RDD":>10} {"BroadcastJoin":>14} {"SortMerge":>12}')
print('-'*65)
print(f'{"Execution Time":<25} {rdd_time:>9.2f}s {bj_time:>13.2f}s {smj_time:>11.2f}s')
print(f'{"Shuffle Required":<25} {"No":>10} {"No":>14} {"Yes":>12}')
print(f'{"Sort Step":<25} {"No":>10} {"No":>14} {"Yes":>12}')
print('='*65)
print('KEY INSIGHT: Broadcast join avoids shuffle entirely.')
print('Sort-Merge join sorts both sides before joining — more expensive.')

Metric                           RDD  BroadcastJoin    SortMerge
-----------------------------------------------------------------
Execution Time                88.03s         11.18s       15.55s
Shuffle Required                  No             No          Yes
Sort Step                         No             No          Yes
KEY INSIGHT: Broadcast join avoids shuffle entirely.
Sort-Merge join sorts both sides before joining — more expensive.
